In [1]:
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import yfinance as yf
from fredapi import Fred
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.covariance import LedoitWolf
from hmmlearn.hmm import GaussianHMM
from joblib import dump, load
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from scipy import linalg as la
from scipy.stats import norm, chi2
from typing import Dict, List, Tuple, Optional
import logging

In [3]:
warnings.filterwarnings('ignore')

In [5]:
os.environ["FRED_API_KEY"] = "81e592493abfdff68010e8d58ce3c1b1"

In [7]:
class Config:
    FRED_KEY = os.getenv("FRED_API_KEY")
    print(f"Value of FRED_API_KEY: {FRED_KEY}") # Add this line
    if not FRED_KEY:
        raise RuntimeError("Set FRED_API_KEY in environment.")
    
    OUTDIR = "out"
    os.makedirs(OUTDIR, exist_ok=True)
    
    # Data range
    START = "1990-01-01"
    END = datetime.today().strftime("%Y-%m-%d")
    
    # Expanded FRED series with more granular data
    FRED_SERIES = {
        # Inflation indicators
        "CPI": "CPIAUCSL",
        "CoreCPI": "CPILFESL",
        "PCE": "PCEPI",
        "CorePCE": "PCEPILFE",
        "BreakevenInfl_5Y": "T5YIE",
        "BreakevenInfl_10Y": "T10YIE",
        
        # Growth indicators
        "GDP": "GDPC1",
        "GDPNow": "GDPNOW",  # Atlanta Fed nowcast if available
        "IndustrialProd": "INDPRO",
        "CapacityUtil": "TCU",
        "RetailSales": "RSAFS",
        "PersonalIncome": "PI",
        "PersonalConsumption": "PCE",
        
        # Labor market
        "Unemp": "UNRATE",
        "Payrolls": "PAYEMS",
        "InitialClaims": "ICSA",
        "ContinuingClaims": "CCSA",
        "LaborForcePartic": "CIVPART",
        "AvgHourlyEarnings": "CES0500000003",
        "JOLTS": "JTSJOL",
        
        # Housing
        "HousingStarts": "HOUST",
        "BuildingPermits": "PERMIT",
        "ExistingHomeSales": "EXHOSLUSM495S",
        "CaseShiller": "CSUSHPISA",
        
        # Money & Credit
        "M2": "M2SL",
        "CommercialBankCredit": "TOTBKCR",
        "ConsumerCredit": "TOTALSL",
        
        # Interest rates
        "FEDFUNDS": "DFF",
        "DGS3MO": "DGS3MO",
        "DGS2": "DGS2",
        "DGS5": "DGS5",
        "DGS10": "DGS10",
        "DGS30": "DGS30",
        "TIPS10": "DFII10",
        
        # Credit spreads
        "AAA_spread": "BAMLC0A0CM",
        "BBB_spread": "BAMLC0A4CBBB",
        "HY_spread": "BAMLH0A0HYM2",
        "TED_spread": "TEDRATE",
        
        # Volatility & Sentiment
        "VIX": "VIXCLS",
        "ConsumerSentiment": "UMCSENT",
        "PMI_Manufacturing": "MANEMP",  # proxy
        
        # Dollar & Commodities
        "DXY": "DTWEXBGS",  # Trade weighted dollar
        "WTI": "DCOILWTICO",
        "Gold": "GOLDAMGBD228NLBM",
    }
    
    # Market tickers for cross-asset analysis
    MARKET_TICKERS = ["SPY", "TLT", "GLD", "DBC", "FXE", "EEM", "HYG", "VNQ"]
    
    # Model parameters
    TARGET_VARS = ["CPI_YoY", "GDP_Growth_q", "Unemp", "DGS10", "DGS2", "SPY_ret_21d"]
    N_FACTORS = 7  # Increased for richer factor structure
    VAR_LAGS = 4  # More lags for quarterly effects
    
    # Advanced parameters
    USE_BAYESIAN_VAR = True
    MINNESOTA_PRIOR_TIGHTNESS = 0.2
    USE_TVP_VAR = False  # Time-varying parameters
    BOOTSTRAP_REPS = 1000
    
    # Backtesting
    BACKTEST_START = "2000-01-01"
    WALK_FORWARD_WINDOW = 252 * 3  # 3 years
    REFIT_FREQUENCY = 63  # Quarterly

# ========== LOGGING SETUP ==========
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler(f"{Config.OUTDIR}/pipeline.log"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

Value of FRED_API_KEY: 81e592493abfdff68010e8d58ce3c1b1


In [9]:
# ========== ENHANCED DATA FETCHING ==========
class DataFetcher:
    """Enhanced data fetching with error handling and caching"""
    
    def __init__(self, config: Config):
        self.config = config
        self.fred = Fred(api_key=config.FRED_KEY)
        self.cache = {}
        
    def fetch_fred_with_fallback(self, series_map: Dict, start: str, end: str) -> pd.DataFrame:
        """Fetch FRED data with fallback handling for missing series"""
        frames = {}
        failed = []
        
        for name, code in series_map.items():
            try:
                if name in self.cache:
                    s = self.cache[name]
                else:
                    s = self.fred.get_series(code, observation_start=start, observation_end=end)
                    s.index = pd.to_datetime(s.index)
                    s.name = name
                    self.cache[name] = s
                
                frames[name] = s
                logger.info(f"Fetched {name} ({code}) rows={len(s)}")
                
            except Exception as e:
                logger.warning(f"Failed to fetch {name} ({code}): {e}")
                failed.append(name)
        
        if not frames:
            raise ValueError("No data successfully fetched")
        
        df = pd.concat(frames.values(), axis=1)
        
        # Sophisticated reindexing for mixed frequencies
        idx = pd.date_range(df.index.min(), df.index.max(), freq='B')
        df = df.reindex(idx)
        
        # Handle different frequencies appropriately
        for col in df.columns:
            if col in ['GDP', 'GDPNow']:  # Quarterly
                df[col] = df[col].interpolate(method='time', limit=65)
            elif col in ['JOLTS', 'ConsumerSentiment']:  # Monthly
                df[col] = df[col].interpolate(method='time', limit=22)
            else:  # Daily/Weekly
                df[col] = df[col].ffill(limit=5)
        
        return df, failed
    
    def fetch_market_data(self, tickers: List[str], start: str, end: str) -> pd.DataFrame:
        """Fetch market data with volume and returns"""
        data = yf.download(tickers, start=start, end=end, progress=False, auto_adjust=True)
        
        prices = data['Close'] if len(tickers) > 1 else data['Close'].to_frame(tickers[0])
        volumes = data['Volume'] if 'Volume' in data else None
        
        prices.index = pd.to_datetime(prices.index)
        prices = prices.resample('B').last().ffill()
        
        # Calculate returns at multiple horizons
        returns = pd.DataFrame(index=prices.index)
        for ticker in prices.columns:
            for h in [1, 5, 21, 63, 252]:
                returns[f"{ticker}_ret_{h}d"] = prices[ticker].pct_change(h)
        
        result = pd.concat([prices, returns], axis=1)
        
        if volumes is not None:
            volumes = volumes.resample('B').last().ffill()
            result = pd.concat([result, volumes.add_suffix('_vol')], axis=1)
        
        return result


In [11]:
# ========== ENHANCED TRANSFORMATIONS ==========
class FeatureEngineer:
    """Advanced feature engineering"""
    
    @staticmethod
    def compute_growth_rates(df: pd.DataFrame) -> pd.DataFrame:
        """Compute various growth rates and transformations"""
        result = pd.DataFrame(index=df.index)
        
        # Year-over-year for monthly series
        monthly_cols = ['CPI', 'CoreCPI', 'PCE', 'CorePCE', 'Payrolls', 
                       'IndustrialProd', 'RetailSales', 'HousingStarts', 'M2']
        for col in monthly_cols:
            if col in df.columns:
                result[f"{col}_YoY"] = df[col].pct_change(252)
                result[f"{col}_MoM_ann"] = df[col].pct_change(21) * 12
        
        # Quarter-over-quarter for GDP
        if 'GDP' in df.columns:
            quarterly = df['GDP'].resample('Q').last()
            qoq = quarterly.pct_change(1) * 4  # Annualized
            result['GDP_Growth_q'] = qoq.reindex(df.index, method='ffill')
        
        # Spreads and curves
        if 'DGS10' in df.columns and 'DGS2' in df.columns:
            result['Curve_10_2'] = df['DGS10'] - df['DGS2']
            result['Curve_10_2_MA'] = result['Curve_10_2'].rolling(21).mean()
        
        if 'DGS10' in df.columns and 'DGS3MO' in df.columns:
            result['Curve_10_3m'] = df['DGS10'] - df['DGS3MO']
        
        # Real rates
        if 'DGS10' in df.columns and 'BreakevenInfl_10Y' in df.columns:
            result['RealRate_10Y'] = df['DGS10'] - df['BreakevenInfl_10Y']
        
        # Credit conditions
        if 'AAA_spread' in df.columns and 'BBB_spread' in df.columns:
            result['Credit_Quality_Spread'] = df['BBB_spread'] - df['AAA_spread']
        
        # Labor market tightness
        if 'JOLTS' in df.columns and 'Unemp' in df.columns:
            result['Labor_Tightness'] = df['JOLTS'] / (df['Unemp'] * 1000)
        
        # Momentum indicators
        for col in ['CPI_YoY', 'GDP_Growth_q', 'DGS10']:
            if col in result.columns:
                result[f"{col}_momentum"] = result[col] - result[col].rolling(63).mean()
        
        return result
    
    @staticmethod
    def compute_technical_indicators(prices: pd.DataFrame) -> pd.DataFrame:
        """Add technical indicators for market data"""
        result = pd.DataFrame(index=prices.index)
        
        for col in prices.columns:
            if '_ret_' not in col and '_vol' not in col:
                # Moving averages
                result[f"{col}_MA50"] = prices[col].rolling(50).mean()
                result[f"{col}_MA200"] = prices[col].rolling(200).mean()
                
                # RSI
                delta = prices[col].diff()
                gain = (delta.where(delta > 0, 0)).rolling(14).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
                rs = gain / loss
                result[f"{col}_RSI"] = 100 - (100 / (1 + rs))
                
                # Bollinger Bands
                ma20 = prices[col].rolling(20).mean()
                std20 = prices[col].rolling(20).std()
                result[f"{col}_BB_upper"] = ma20 + 2 * std20
                result[f"{col}_BB_lower"] = ma20 - 2 * std20
                result[f"{col}_BB_width"] = result[f"{col}_BB_upper"] - result[f"{col}_BB_lower"]
        
        return result

In [13]:
class BayesianVAR:
    """Bayesian VAR with Minnesota prior"""
    
    def __init__(self, lags: int = 3, prior_tightness: float = 0.2):
        self.lags = lags
        self.prior_tightness = prior_tightness
        self.coefs = None
        self.sigma = None
        
    def fit(self, Y: np.ndarray) -> Dict:
        """Fit Bayesian VAR with Minnesota prior"""
        T, n = Y.shape
        p = self.lags
        logger.info(f"Y shape: {Y.shape}, lags: {p}")
        
        # Create lagged matrix
        Yt, X = self._make_lagged_matrix(Y, p)
        X = np.hstack([np.ones((X.shape[0], 1)), X])  # Add intercept
        logger.info(f"Yt shape: {Yt.shape}, X shape: {X.shape}")
        
        # Minnesota prior setup
        lambda1 = self.prior_tightness  # Overall tightness
        lambda2 = 1.0  # Cross-variable tightness
        lambda3 = 1.0  # Lag decay
        
        # Prior mean (random walk for each variable)
        B0 = np.zeros((X.shape[1], n))  # Shape: (n*p + 1, n)
        for i in range(n):
            B0[1 + i, i] = 1.0  # First own lag = 1
        
        # Prior variance
        sigma_sq = np.var(Y, axis=0)
        V0 = np.zeros((X.shape[1], X.shape[1]))  # Shape: (n*p + 1, n*p + 1)
        V0[0, 0] = 1e6  # Loose prior on intercept
        
        for lag in range(p):
            for i in range(n):
                for j in range(n):
                    idx = 1 + lag * n + j
                    if i == j:
                        V0[idx, idx] = (lambda1 / (lag + 1) ** lambda3) ** 2
                    else:
                        V0[idx, idx] = (lambda1 * lambda2 / (lag + 1) ** lambda3) ** 2 * (sigma_sq[i] / sigma_sq[j])
        
        # Posterior parameters (conjugate normal)
        V0_inv = np.linalg.pinv(V0)  # Use pinv for numerical stability
        Vpost_inv = V0_inv + X.T @ X
        Vpost = np.linalg.pinv(Vpost_inv)  # Use pinv for numerical stability
        
        # Compute posterior mean
        Bpost = Vpost @ (V0_inv @ B0 + X.T @ Yt)
        Bpost = Bpost.reshape(X.shape[1], n)
        
        # Extract coefficients
        self.intercept = Bpost[0, :]
        self.A = []
        for lag in range(p):
            self.A.append(Bpost[1 + lag * n:1 + (lag + 1) * n, :].T)
        
        # Residual covariance
        residuals = Yt - X @ Bpost
        self.sigma = np.cov(residuals.T)
        
        return {
            "intercept": self.intercept,
            "A": self.A,
            "sigma_u": self.sigma,
            "residuals": residuals,
            "posterior_mean": Bpost,
            "posterior_cov": Vpost
        }
    
    def _make_lagged_matrix(self, Y: np.ndarray, p: int) -> Tuple[np.ndarray, np.ndarray]:
        """Create lagged matrices for VAR"""
        T, n = Y.shape
        rows = T - p
        X = np.zeros((rows, n * p))
        for i in range(p):
            X[:, i * n:(i + 1) * n] = Y[p - i - 1:T - i - 1, :]
        Yt = Y[p:, :]
        return Yt, X

In [15]:
# ========== BAYESIAN VAR IMPLEMENTATION ==========
class BayesianVAR:
    """Bayesian VAR with Minnesota prior"""
    
    def __init__(self, lags: int = 3, prior_tightness: float = 0.2):
        self.lags = lags
        self.prior_tightness = prior_tightness
        self.coefs = None
        self.sigma = None
        
    def fit(self, Y: np.ndarray) -> Dict:
        """Fit Bayesian VAR with Minnesota prior"""
        T, n = Y.shape
        p = self.lags
        
        # Create lagged matrix
        Yt, X = self._make_lagged_matrix(Y, p)
        X = np.hstack([np.ones((X.shape[0], 1)), X])  # Add intercept
        
        # Minnesota prior setup
        lambda1 = self.prior_tightness  # Overall tightness
        lambda2 = 1.0  # Cross-variable tightness
        lambda3 = 1.0  # Lag decay
        
        # Prior mean (random walk for each variable)
        B0 = np.zeros((X.shape[1], n))
        for i in range(n):
            B0[1 + i, i] = 1.0  # First own lag = 1
        
        # Prior variance
        sigma_sq = np.var(Y, axis=0)
        V0 = np.zeros((X.shape[1], X.shape[1]))
        V0[0, 0] = 1e6  # Loose prior on intercept
        
        for lag in range(p):
            for i in range(n):
                for j in range(n):
                    idx = 1 + lag * n + j
                    if i == j:
                        V0[idx, idx] = (lambda1 / (lag + 1) ** lambda3) ** 2
                    else:
                        V0[idx, idx] = (lambda1 * lambda2 / (lag + 1) ** lambda3) ** 2 * (sigma_sq[i] / sigma_sq[j])
        
        # Posterior parameters (conjugate normal)
        V0_inv = np.linalg.inv(V0)
        Vpost_inv = V0_inv + X.T @ X
        Vpost = np.linalg.inv(Vpost_inv)
        Bpost = Vpost @ (V0_inv @ B0.flatten() + X.T @ Yt.flatten())
        Bpost = Bpost.reshape(X.shape[1], n)
        
        # Extract coefficients
        self.intercept = Bpost[0, :]
        self.A = []
        for lag in range(p):
            self.A.append(Bpost[1 + lag * n:1 + (lag + 1) * n, :].T)
        
        # Residual covariance
        residuals = Yt - X @ Bpost
        self.sigma = np.cov(residuals.T)

In [17]:
# ========== ENHANCED TRANSFORMATIONS ==========
class FeatureEngineer:
    """Advanced feature engineering"""
    
    @staticmethod
    def compute_growth_rates(df: pd.DataFrame) -> pd.DataFrame:
        """Compute various growth rates and transformations"""
        result = pd.DataFrame(index=df.index)
        
        # Year-over-year for monthly series
        monthly_cols = ['CPI', 'CoreCPI', 'PCE', 'CorePCE', 'Payrolls', 
                       'IndustrialProd', 'RetailSales', 'HousingStarts', 'M2']
        for col in monthly_cols:
            if col in df.columns:
                result[f"{col}_YoY"] = df[col].pct_change(252)
                result[f"{col}_MoM_ann"] = df[col].pct_change(21) * 12
        
        # Quarter-over-quarter for GDP
        if 'GDP' in df.columns:
            quarterly = df['GDP'].resample('Q').last()
            qoq = quarterly.pct_change(1) * 4  # Annualized
            result['GDP_Growth_q'] = qoq.reindex(df.index, method='ffill')
        
        # Spreads and curves
        if 'DGS10' in df.columns and 'DGS2' in df.columns:
            result['Curve_10_2'] = df['DGS10'] - df['DGS2']
            result['Curve_10_2_MA'] = result['Curve_10_2'].rolling(21).mean()
        
        if 'DGS10' in df.columns and 'DGS3MO' in df.columns:
            result['Curve_10_3m'] = df['DGS10'] - df['DGS3MO']
        
        # Real rates
        if 'DGS10' in df.columns and 'BreakevenInfl_10Y' in df.columns:
            result['RealRate_10Y'] = df['DGS10'] - df['BreakevenInfl_10Y']
        
        # Credit conditions
        if 'AAA_spread' in df.columns and 'BBB_spread' in df.columns:
            result['Credit_Quality_Spread'] = df['BBB_spread'] - df['AAA_spread']
        
        # Labor market tightness
        if 'JOLTS' in df.columns and 'Unemp' in df.columns:
            result['Labor_Tightness'] = df['JOLTS'] / (df['Unemp'] * 1000)
        
        # Momentum indicators
        for col in ['CPI_YoY', 'GDP_Growth_q', 'DGS10']:
            if col in result.columns:
                result[f"{col}_momentum"] = result[col] - result[col].rolling(63).mean()
        
        return result
    
    @staticmethod
    def compute_technical_indicators(prices: pd.DataFrame) -> pd.DataFrame:
        """Add technical indicators for market data"""
        result = pd.DataFrame(index=prices.index)
        
        for col in prices.columns:
            if '_ret_' not in col and '_vol' not in col:
                # Moving averages
                result[f"{col}_MA50"] = prices[col].rolling(50).mean()
                result[f"{col}_MA200"] = prices[col].rolling(200).mean()
                
                # RSI
                delta = prices[col].diff()
                gain = (delta.where(delta > 0, 0)).rolling(14).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
                rs = gain / loss
                result[f"{col}_RSI"] = 100 - (100 / (1 + rs))
                
                # Bollinger Bands
                ma20 = prices[col].rolling(20).mean()
                std20 = prices[col].rolling(20).std()
                result[f"{col}_BB_upper"] = ma20 + 2 * std20
                result[f"{col}_BB_lower"] = ma20 - 2 * std20
                result[f"{col}_BB_width"] = result[f"{col}_BB_upper"] - result[f"{col}_BB_lower"]
        
        return result


In [19]:
# ========== BAYESIAN VAR IMPLEMENTATION ==========
class BayesianVAR:
    """Bayesian VAR with Minnesota prior"""
    
    def __init__(self, lags: int = 3, prior_tightness: float = 0.2):
        self.lags = lags
        self.prior_tightness = prior_tightness
        self.coefs = None
        self.sigma = None
        
    def fit(self, Y: np.ndarray) -> Dict:
        """Fit Bayesian VAR with Minnesota prior"""
        T, n = Y.shape
        p = self.lags
        
        # Create lagged matrix
        Yt, X = self._make_lagged_matrix(Y, p)
        X = np.hstack([np.ones((X.shape[0], 1)), X])  # Add intercept
        
        # Minnesota prior setup
        lambda1 = self.prior_tightness  # Overall tightness
        lambda2 = 1.0  # Cross-variable tightness
        lambda3 = 1.0  # Lag decay
        
        # Prior mean (random walk for each variable)
        B0 = np.zeros((X.shape[1], n))
        for i in range(n):
            B0[1 + i, i] = 1.0  # First own lag = 1
        
        # Prior variance
        sigma_sq = np.var(Y, axis=0)
        V0 = np.zeros((X.shape[1], X.shape[1]))
        V0[0, 0] = 1e6  # Loose prior on intercept
        
        for lag in range(p):
            for i in range(n):
                for j in range(n):
                    idx = 1 + lag * n + j
                    if i == j:
                        V0[idx, idx] = (lambda1 / (lag + 1) ** lambda3) ** 2
                    else:
                        V0[idx, idx] = (lambda1 * lambda2 / (lag + 1) ** lambda3) ** 2 * (sigma_sq[i] / sigma_sq[j])
        
        # Posterior parameters (conjugate normal)
        V0_inv = np.linalg.inv(V0)
        Vpost_inv = V0_inv + X.T @ X
        Vpost = np.linalg.inv(Vpost_inv)
        Bpost = Vpost @ (V0_inv @ B0.flatten() + X.T @ Yt.flatten())
        Bpost = Bpost.reshape(X.shape[1], n)
        
        # Extract coefficients
        self.intercept = Bpost[0, :]
        self.A = []
        for lag in range(p):
            self.A.append(Bpost[1 + lag * n:1 + (lag + 1) * n, :].T)
        
        # Residual covariance
        residuals = Yt - X @ Bpost
        self.sigma = np.cov(residuals.T)

In [21]:

# ========== ENHANCED TRANSFORMATIONS ==========
class FeatureEngineer:
    """Advanced feature engineering"""
    
    @staticmethod
    def compute_growth_rates(df: pd.DataFrame) -> pd.DataFrame:
        """Compute various growth rates and transformations"""
        result = pd.DataFrame(index=df.index)
        
        # Year-over-year for monthly series
        monthly_cols = ['CPI', 'CoreCPI', 'PCE', 'CorePCE', 'Payrolls', 
                       'IndustrialProd', 'RetailSales', 'HousingStarts', 'M2']
        for col in monthly_cols:
            if col in df.columns:
                result[f"{col}_YoY"] = df[col].pct_change(252)
                result[f"{col}_MoM_ann"] = df[col].pct_change(21) * 12
        
        # Quarter-over-quarter for GDP
        if 'GDP' in df.columns:
            quarterly = df['GDP'].resample('Q').last()
            qoq = quarterly.pct_change(1) * 4  # Annualized
            result['GDP_Growth_q'] = qoq.reindex(df.index, method='ffill')
        
        # Spreads and curves
        if 'DGS10' in df.columns and 'DGS2' in df.columns:
            result['Curve_10_2'] = df['DGS10'] - df['DGS2']
            result['Curve_10_2_MA'] = result['Curve_10_2'].rolling(21).mean()
        
        if 'DGS10' in df.columns and 'DGS3MO' in df.columns:
            result['Curve_10_3m'] = df['DGS10'] - df['DGS3MO']
        
        # Real rates
        if 'DGS10' in df.columns and 'BreakevenInfl_10Y' in df.columns:
            result['RealRate_10Y'] = df['DGS10'] - df['BreakevenInfl_10Y']
        
        # Credit conditions
        if 'AAA_spread' in df.columns and 'BBB_spread' in df.columns:
            result['Credit_Quality_Spread'] = df['BBB_spread'] - df['AAA_spread']
        
        # Labor market tightness
        if 'JOLTS' in df.columns and 'Unemp' in df.columns:
            result['Labor_Tightness'] = df['JOLTS'] / (df['Unemp'] * 1000)
        
        # Momentum indicators
        for col in ['CPI_YoY', 'GDP_Growth_q', 'DGS10']:
            if col in result.columns:
                result[f"{col}_momentum"] = result[col] - result[col].rolling(63).mean()
        
        return result
    
    @staticmethod
    def compute_technical_indicators(prices: pd.DataFrame) -> pd.DataFrame:
        """Add technical indicators for market data"""
        result = pd.DataFrame(index=prices.index)
        
        for col in prices.columns:
            if '_ret_' not in col and '_vol' not in col:
                # Moving averages
                result[f"{col}_MA50"] = prices[col].rolling(50).mean()
                result[f"{col}_MA200"] = prices[col].rolling(200).mean()
                
                # RSI
                delta = prices[col].diff()
                gain = (delta.where(delta > 0, 0)).rolling(14).mean()
                loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
                rs = gain / loss
                result[f"{col}_RSI"] = 100 - (100 / (1 + rs))
                
                # Bollinger Bands
                ma20 = prices[col].rolling(20).mean()
                std20 = prices[col].rolling(20).std()
                result[f"{col}_BB_upper"] = ma20 + 2 * std20
                result[f"{col}_BB_lower"] = ma20 - 2 * std20
                result[f"{col}_BB_width"] = result[f"{col}_BB_upper"] - result[f"{col}_BB_lower"]
        
        return result


In [23]:

# ========== BAYESIAN VAR IMPLEMENTATION ==========
class BayesianVAR:
    """Bayesian VAR with Minnesota prior"""
    
    def __init__(self, lags: int = 3, prior_tightness: float = 0.2):
        self.lags = lags
        self.prior_tightness = prior_tightness
        self.coefs = None
        self.sigma = None
        
    def fit(self, Y: np.ndarray) -> Dict:
        """Fit Bayesian VAR with Minnesota prior"""
        T, n = Y.shape
        p = self.lags
        
        # Create lagged matrix
        Yt, X = self._make_lagged_matrix(Y, p)
        X = np.hstack([np.ones((X.shape[0], 1)), X])  # Add intercept
        
        # Minnesota prior setup
        lambda1 = self.prior_tightness  # Overall tightness
        lambda2 = 1.0  # Cross-variable tightness
        lambda3 = 1.0  # Lag decay
        
        # Prior mean (random walk for each variable)
        B0 = np.zeros((X.shape[1], n))
        for i in range(n):
            B0[1 + i, i] = 1.0  # First own lag = 1
        
        # Prior variance
        sigma_sq = np.var(Y, axis=0)
        V0 = np.zeros((X.shape[1], X.shape[1]))
        V0[0, 0] = 1e6  # Loose prior on intercept
        
        for lag in range(p):
            for i in range(n):
                for j in range(n):
                    idx = 1 + lag * n + j
                    if i == j:
                        V0[idx, idx] = (lambda1 / (lag + 1) ** lambda3) ** 2
                    else:
                        V0[idx, idx] = (lambda1 * lambda2 / (lag + 1) ** lambda3) ** 2 * (sigma_sq[i] / sigma_sq[j])
        
        # Posterior parameters (conjugate normal)
        V0_inv = np.linalg.inv(V0)
        Vpost_inv = V0_inv + X.T @ X
        Vpost = np.linalg.inv(Vpost_inv)
        Bpost = Vpost @ (V0_inv @ B0.flatten() + X.T @ Yt.flatten())
        Bpost = Bpost.reshape(X.shape[1], n)
        
        # Extract coefficients
        self.intercept = Bpost[0, :]
        self.A = []
        for lag in range(p):
            self.A.append(Bpost[1 + lag * n:1 + (lag + 1) * n, :].T)
        
        # Residual covariance
        residuals = Yt - X @ Bpost
        self.sigma = np.cov(residuals.T)
        
        return {
            "intercept": self.intercept,
            "A": self.A,
            "sigma_u": self.sigma,
            "residuals": residuals,
            "posterior_mean": Bpost,
            "posterior_cov": Vpost
        }
    
    def _make_lagged_matrix(self, Y: np.ndarray, p: int) -> Tuple[np.ndarray, np.ndarray]:
        T, n = Y.shape
        rows = T - p
        X = np.zeros((rows, n * p))
        for i in range(p):
            X[:, i * n:(i + 1) * n] = Y[p - i - 1:T - i - 1, :]
        Yt = Y[p:, :]
        return Yt, X


In [25]:

# ========== ADVANCED FACTOR MODELS ==========
class DynamicFactorModel:
    """Dynamic factor model with EM algorithm"""
    
    def __init__(self, n_factors: int = 5, lags: int = 2):
        self.n_factors = n_factors
        self.lags = lags
        self.loadings = None
        self.factors = None
        
    def fit(self, X: np.ndarray, max_iter: int = 100) -> np.ndarray:
        """Fit dynamic factor model using EM algorithm"""
        T, N = X.shape
        K = self.n_factors
        
        # Initialize with PCA
        pca = PCA(n_components=K)
        F_init = pca.fit_transform(X)
        Lambda_init = pca.components_.T
        
        # EM iterations
        F = F_init.copy()
        Lambda = Lambda_init.copy()
        
        for iter in range(max_iter):
            # E-step: estimate factors given loadings
            F_new = X @ Lambda @ np.linalg.inv(Lambda.T @ Lambda)
            
            # M-step: estimate loadings given factors
            Lambda_new = np.linalg.inv(F_new.T @ F_new) @ F_new.T @ X
            Lambda_new = Lambda_new.T
            
            # Check convergence
            if np.max(np.abs(F_new - F)) < 1e-4:
                break
            
            F = F_new
            Lambda = Lambda_new
        
        self.factors = F
        self.loadings = Lambda
        
        return F


In [27]:

# ========== IRF WITH CONFIDENCE BANDS ==========
class IRFAnalysis:
    """Enhanced IRF analysis with bootstrapped confidence intervals"""
    
    @staticmethod
    def compute_irf_with_bands(var_res: Dict, horizon: int = 36, 
                               bootstrap: int = 1000, conf_level: float = 0.95) -> Dict:
        """Compute IRFs with bootstrapped confidence bands"""
        
        # Point estimate IRF
        irfs = IRFAnalysis._compute_irf_point(var_res, horizon)
        
        # Bootstrap for confidence bands
        n = var_res["sigma_u"].shape[0]
        irf_samples = np.zeros((bootstrap, horizon + 1, n, n))
        
        residuals = var_res["residuals"]
        T = residuals.shape[0]
        
        for b in range(bootstrap):
            # Resample residuals
            idx = np.random.choice(T, T, replace=True)
            res_boot = residuals[idx, :]
            
            # Refit VAR (simplified - just perturb coefficients)
            A_boot = []
            for A_mat in var_res["A"]:
                noise = np.random.normal(0, 0.01, A_mat.shape)
                A_boot.append(A_mat + noise)
            
            var_boot = {
                "A": A_boot,
                "sigma_u": np.cov(res_boot.T),
                "intercept": var_res["intercept"]
            }
            
            irf_samples[b] = IRFAnalysis._compute_irf_point(var_boot, horizon)
        
        # Compute confidence bands
        alpha = 1 - conf_level
        lower = np.percentile(irf_samples, 100 * alpha / 2, axis=0)
        upper = np.percentile(irf_samples, 100 * (1 - alpha / 2), axis=0)
        
        return {
            "irf": irfs,
            "lower": lower,
            "upper": upper,
            "samples": irf_samples
        }
    
    @staticmethod
    def _compute_irf_point(var_res: Dict, horizon: int) -> np.ndarray:
        """Compute point estimate of IRF"""
        A = var_res["A"]
        sigma = var_res["sigma_u"]
        p = len(A)
        n = A[0].shape[0]
        
        # Companion matrix
        F = IRFAnalysis._companion_matrix(A)
        
        # Cholesky for orthogonalization
        P = la.cholesky(sigma, lower=True)
        
        # Compute IRF
        irfs = np.zeros((horizon + 1, n, n))
        irfs[0, :, :] = P
        
        comp_power = np.eye(n * p)
        for h in range(1, horizon + 1):
            comp_power = comp_power @ F
            M = comp_power[:n, :n]
            irfs[h, :, :] = M @ P
        
        return irfs
    
    @staticmethod
    def _companion_matrix(A_list: List[np.ndarray]) -> np.ndarray:
        p = len(A_list)
        n = A_list[0].shape[0]
        
        top = np.hstack(A_list)
        if p == 1:
            bottom = np.zeros((0, n * p))
        else:
            bottom = np.eye(n * (p - 1), n * p)
        
        return np.vstack([top, bottom])


In [29]:

# ========== REGIME SWITCHING WITH MARKOV CHAINS ==========
class RegimeSwitchingVAR:
    """VAR with Markov-switching regimes"""
    
    def __init__(self, n_regimes: int = 3, lags: int = 3):
        self.n_regimes = n_regimes
        self.lags = lags
        self.regime_vars = {}
        self.transition_matrix = None
        
    def fit(self, Y: pd.DataFrame) -> Dict:
        """Fit regime-switching VAR"""
        
        # First identify regimes using HMM on factors
        hmm = GaussianHMM(n_components=self.n_regimes, covariance_type='full', 
                          n_iter=500, random_state=42)
        
        Y_scaled = StandardScaler().fit_transform(Y)
        hmm.fit(Y_scaled)
        regimes = hmm.predict(Y_scaled)
        
        # Estimate VAR for each regime
        for r in range(self.n_regimes):
            mask = regimes == r
            if mask.sum() > self.lags * Y.shape[1] * 2:  # Enough data
                Y_regime = Y[mask]
                var = BayesianVAR(lags=self.lags)
                self.regime_vars[r] = var.fit(Y_regime.values)
        
        # Estimate transition probabilities
        self.transition_matrix = hmm.transmat_
        
        return {
            "regimes": regimes,
            "regime_vars": self.regime_vars,
            "transition_matrix": self.transition_matrix,
            "hmm_model": hmm
        }


In [31]:
class Nowcaster:
    """Real-time nowcasting using mixed-frequency data"""
    
    def __init__(self, target: str = "GDP_Growth_q"):
        self.target = target
        self.model = None
        
    def fit(self, high_freq: pd.DataFrame, low_freq: pd.Series) -> None:
        """Fit nowcasting model using MIDAS or state-space approach"""
        # Ensure low_freq has quarterly index
        low_freq = low_freq.resample('Q').last().dropna()
        logger.info(f"Low-freq index: {low_freq.index[:5]}")
        logger.info(f"High-freq columns: {high_freq.columns.tolist()}")
        logger.info(f"High-freq index: {high_freq.index[:5]}")
        
        # Align data
        aligned = pd.DataFrame(index=low_freq.index)
        
        for col in high_freq.columns:
            # Resample high-freq data to quarterly frequency, using last available value
            resampled = high_freq[col].resample('Q').last()
            # Fill missing values before reindexing
            resampled = resampled.fillna(method='ffill').fillna(method='bfill')
            aligned[col] = resampled.reindex(low_freq.index, method='ffill')
        
        aligned[self.target] = low_freq
        logger.info(f"Aligned shape before dropna: {aligned.shape}")
        logger.info(f"Aligned head:\n{aligned.head()}")
        
        aligned = aligned.dropna()
        if aligned.empty:
            logger.error(f"Aligned DataFrame is empty after dropna. Check data availability for high_freq columns.")
            raise ValueError("No overlapping data after alignment. Ensure high_freq columns have data for low_freq periods.")
        
        logger.info(f"Aligned shape after dropna: {aligned.shape}")
        
        # Fit elastic net for variable selection
        X = aligned.drop(columns=[self.target])
        y = aligned[self.target]
        
        self.model = ElasticNet(alpha=0.1, l1_ratio=0.5)
        self.model.fit(X, y)
        
        # Store feature importance
        self.feature_importance = pd.Series(
            np.abs(self.model.coef_), 
            index=X.columns
        ).sort_values(ascending=False)
    
    def nowcast(self, current_high_freq: pd.DataFrame) -> float:
        """Generate nowcast for current period"""
        if self.model is None:
            raise ValueError("Model not fitted")
        
        # Use latest available data, resampled to quarterly
        X_now = current_high_freq.resample('Q').last().fillna(method='ffill').iloc[-1:].values
        nowcast = self.model.predict(X_now)[0]
        
        return nowcast

In [33]:

# ========== BACKTESTING FRAMEWORK ==========
class Backtester:
    """Walk-forward backtesting with performance metrics"""
    
    def __init__(self, config: Config):
        self.config = config
        self.results = []
        
    def run_backtest(self, data: pd.DataFrame, model_func, 
                     start_date: str, end_date: str) -> pd.DataFrame:
        """Run walk-forward backtest"""
        
        window = self.config.WALK_FORWARD_WINDOW
        refit_freq = self.config.REFIT_FREQUENCY
        
        test_dates = pd.date_range(start_date, end_date, freq='B')
        
        predictions = []
        actuals = []
        
        for i, test_date in enumerate(test_dates):
            if i % refit_freq == 0:  # Refit model
                train_end = test_date - timedelta(days=1)
                train_start = train_end - timedelta(days=window)
                
                train_data = data.loc[train_start:train_end]
                if len(train_data) < 100:
                    continue
                
                model = model_func(train_data)
            
            # Make prediction
            try:
                pred = model.predict(data.loc[:test_date])
                predictions.append(pred)
                
                # Get actual (if available)
                if test_date in data.index:
                    actuals.append(data.loc[test_date])
            except:
                continue
        
        # Calculate performance metrics
        results = pd.DataFrame({
            'prediction': predictions,
            'actual': actuals[:len(predictions)]
        })
        
        metrics = self._calculate_metrics(results)
        
        return results, metrics
    
    def _calculate_metrics(self, results: pd.DataFrame) -> Dict:
        """Calculate backtest performance metrics"""
        
        errors = results['actual'] - results['prediction']
        
        metrics = {
            'RMSE': np.sqrt(np.mean(errors ** 2)),
            'MAE': np.mean(np.abs(errors)),
            'MAPE': np.mean(np.abs(errors / results['actual'])) * 100,
            'DirectionalAccuracy': np.mean(np.sign(results['prediction']) == np.sign(results['actual'])),
            'R2': 1 - np.var(errors) / np.var(results['actual'])
        }
        
        return metrics


In [35]:


# ========== MAIN PIPELINE ==========
class MacroVARPipeline:
    """Main pipeline orchestrator"""
    
    def __init__(self, config: Config):
        self.config = config
        self.data_fetcher = DataFetcher(config)
        self.feature_engineer = FeatureEngineer()
        self.results = {}
        
    def run(self) -> Dict:
        """Execute full pipeline"""
        
        logger.info("=" * 50)
        logger.info("Starting Enhanced Macro-Financial VAR Pipeline")
        logger.info("=" * 50)
        
        # 1. Data Collection
        logger.info("Step 1: Fetching macro data from FRED...")
        fred_df, failed_series = self.data_fetcher.fetch_fred_with_fallback(
            self.config.FRED_SERIES, 
            self.config.START, 
            self.config.END
        )
        
        if failed_series:
            logger.warning(f"Failed to fetch: {failed_series}")
        
        # 2. Market Data
        logger.info("Step 2: Fetching market data...")
        market_df = self.data_fetcher.fetch_market_data(
            self.config.MARKET_TICKERS,
            self.config.START,
            self.config.END
        )
        
        # 3. Feature Engineering
        logger.info("Step 3: Engineering features...")
        macro_features = self.feature_engineer.compute_growth_rates(fred_df)
        market_features = self.feature_engineer.compute_technical_indicators(market_df)
        
        # Combine all features
        all_features = pd.concat([fred_df, macro_features, market_df, market_features], axis=1)
        all_features = all_features.dropna(how='all').fillna(method='ffill', limit=5)
        
        # 4. Dynamic Factor Model
        logger.info("Step 4: Extracting dynamic factors...")
        
        # Select features for factor model
        factor_cols = [col for col in all_features.columns 
                      if any(x in col for x in ['YoY', 'Growth', 'Curve', 'spread', 'momentum'])]
        
        if factor_cols:
            X_factors = all_features[factor_cols].dropna()
            X_scaled = StandardScaler().fit_transform(X_factors)
            
            dfm = DynamicFactorModel(n_factors=self.config.N_FACTORS)
            factors = dfm.fit(X_scaled)
            
            factors_df = pd.DataFrame(
                factors, 
                index=X_factors.index,
                columns=[f"Factor_{i+1}" for i in range(self.config.N_FACTORS)]
            )
        else:
            # Fallback to simple PCA
            X_scaled = StandardScaler().fit_transform(all_features.select_dtypes(include=[np.number]).dropna())
            pca = PCA(n_components=self.config.N_FACTORS)
            factors = pca.fit_transform(X_scaled)
            factors_df = pd.DataFrame(
                factors,
                index=all_features.select_dtypes(include=[np.number]).dropna().index,
                columns=[f"Factor_{i+1}" for i in range(self.config.N_FACTORS)]
            )
        
        # 5. Prepare VAR dataset
        logger.info("Step 5: Preparing VAR dataset...")
        
        # Select target variables
        var_cols = []
        for col in self.config.TARGET_VARS:
            if col in all_features.columns:
                var_cols.append(col)
            elif col == "SPY_ret_21d" and "SPY_ret_21d" in market_df.columns:
                var_cols.append(col)
        
        var_df = pd.concat([factors_df, all_features[var_cols]], axis=1).dropna()
        var_df = var_df.loc[var_df.index >= pd.to_datetime("1995-01-01")]
        
        # 6. Fit VAR models
        logger.info("Step 6: Fitting VAR models...")
        
        if self.config.USE_BAYESIAN_VAR:
            logger.info("Using Bayesian VAR with Minnesota prior...")
            var_model = BayesianVAR(
                lags=self.config.VAR_LAGS,
                prior_tightness=self.config.MINNESOTA_PRIOR_TIGHTNESS
            )
            var_res = var_model.fit(var_df.values)
        else:
            logger.info("Using penalized VAR...")
            var_res = self._fit_penalized_var(var_df)
        
        # 7. Regime Switching Analysis
        logger.info("Step 7: Identifying regimes...")
        rs_var = RegimeSwitchingVAR(n_regimes=3, lags=self.config.VAR_LAGS)
        regime_results = rs_var.fit(var_df)
        
        # 8. IRF Analysis with Confidence Bands
        logger.info("Step 8: Computing IRFs with confidence bands...")
        irf_results = IRFAnalysis.compute_irf_with_bands(
            var_res, 
            horizon=36,
            bootstrap=min(self.config.BOOTSTRAP_REPS, 500),  # Limit for speed
            conf_level=0.95
        )
        
        # 9. Nowcasting
        logger.info("Step 9: Setting up nowcasting...")
        if 'GDP_Growth_q' in var_df.columns:
            nowcaster = Nowcaster(target='GDP_Growth_q')
            
            # Use high-frequency indicators
            high_freq_cols = ['CPI_YoY', 'Payrolls_YoY', 'InitialClaims', 'RetailSales_YoY']
            high_freq_data = all_features[[c for c in high_freq_cols if c in all_features.columns]]
            
            if len(high_freq_data.columns) > 2:
                low_freq_target = all_features['GDP_Growth_q'].dropna()
                nowcaster.fit(high_freq_data, low_freq_target)
                
                # Current nowcast
                current_nowcast = nowcaster.nowcast(high_freq_data)
                logger.info(f"GDP Growth Nowcast: {current_nowcast:.2%}")
        
        # 10. Scenario Analysis
        logger.info("Step 10: Running scenario analysis...")
        scenarios = self._run_scenario_analysis(var_df, factors_df, all_features)
        
        # 11. Generate Reports and Visualizations
        logger.info("Step 11: Generating outputs...")
        self._generate_outputs(var_res, irf_results, regime_results, scenarios, var_df)
        
        # Store results
        self.results = {
            'var_results': var_res,
            'irf_results': irf_results,
            'regime_results': regime_results,
            'scenarios': scenarios,
            'data': {
                'var_df': var_df,
                'factors': factors_df,
                'all_features': all_features
            }
        }
        
        logger.info("Pipeline completed successfully!")
        return self.results
    
    def _fit_penalized_var(self, var_df: pd.DataFrame) -> Dict:
        """Fit VAR with Ridge/ElasticNet penalty"""
        Y = var_df.values
        T, n = Y.shape
        p = self.config.VAR_LAGS
        
        # Create lagged matrix
        Yt, X = self._make_lagged_matrix(Y, p)
        X = np.hstack([np.ones((X.shape[0], 1)), X])
        
        # Fit with time series cross-validation for hyperparameter selection
        tscv = TimeSeriesSplit(n_splits=3)
        
        coefs = np.zeros((n, X.shape[1]))
        residuals = np.zeros_like(Yt)
        
        for i in range(n):
            # Grid search for best alpha
            model = Ridge()
            param_grid = {'alpha': np.logspace(-3, 3, 20)}
            
            grid = GridSearchCV(model, param_grid, cv=tscv, scoring='neg_mean_squared_error')
            grid.fit(X, Yt[:, i])
            
            best_model = grid.best_estimator_
            coefs[i, :] = best_model.coef_
            residuals[:, i] = Yt[:, i] - best_model.predict(X)
        
        # Extract coefficient matrices
        intercept = coefs[:, 0]
        A = []
        for lag in range(p):
            A.append(coefs[:, 1 + lag*n : 1 + (lag+1)*n].T)
        
        sigma_u = LedoitWolf().fit(residuals).covariance_
        
        return {
            "intercept": intercept,
            "A": A,
            "sigma_u": sigma_u,
            "residuals": residuals
        }
    
    def _make_lagged_matrix(self, Y: np.ndarray, p: int) -> Tuple[np.ndarray, np.ndarray]:
        T, n = Y.shape
        rows = T - p
        X = np.zeros((rows, n * p))
        for i in range(p):
            X[:, i * n:(i + 1) * n] = Y[p - i - 1:T - i - 1, :]
        Yt = Y[p:, :]
        return Yt, X
    
    def _run_scenario_analysis(self, var_df: pd.DataFrame, factors_df: pd.DataFrame, 
                               all_features: pd.DataFrame) -> Dict:
        """Advanced scenario matching and analysis"""
        
        # Build state vector from latest observation
        latest_date = var_df.index[-1]
        
        # Key state variables
        state_cols = []
        for col in ['CPI_YoY', 'GDP_Growth_q', 'Unemp', 'Curve_10_2', 'VIX']:
            if col in all_features.columns:
                state_cols.append(col)
        
        # Add factors
        state_cols.extend([f"Factor_{i+1}" for i in range(min(3, factors_df.shape[1]))])
        
        # Current state
        current_state = pd.concat([
            all_features.loc[latest_date, state_cols[:5]],
            factors_df.loc[latest_date, state_cols[5:]]
        ])
        
        # Historical states
        historical = pd.concat([
            all_features[state_cols[:5]],
            factors_df[state_cols[5:]]
        ], axis=1).dropna()
        
        # Compute Mahalanobis distance
        cov_matrix = LedoitWolf().fit(historical.values).covariance_
        inv_cov = la.pinv(cov_matrix)
        
        diffs = historical.values - current_state.values
        distances = np.sqrt(np.sum(diffs @ inv_cov * diffs, axis=1))
        
        historical['distance'] = distances
        
        # Find similar episodes (exclude recent 6 months)
        cutoff = latest_date - pd.DateOffset(months=6)
        candidates = historical[historical.index < cutoff].sort_values('distance').head(30)
        
        # Analyze forward outcomes
        outcomes = []
        horizons = [21, 63, 126, 252]
        
        for date in candidates.index[:20]:
            outcome = {'date': date, 'distance': candidates.loc[date, 'distance']}
            
            for h in horizons:
                future_date = date + pd.DateOffset(days=h)
                
                # Get market returns if available
                if 'SPY' in all_features.columns and future_date in all_features.index:
                    ret = (all_features.loc[future_date, 'SPY'] / 
                          all_features.loc[date, 'SPY'] - 1)
                    outcome[f'SPY_{h}d'] = ret
                
                # Get macro outcomes
                if 'CPI_YoY' in all_features.columns and future_date in all_features.index:
                    outcome[f'CPI_chg_{h}d'] = (all_features.loc[future_date, 'CPI_YoY'] - 
                                                all_features.loc[date, 'CPI_YoY'])
            
            outcomes.append(outcome)
        
        outcomes_df = pd.DataFrame(outcomes)
        
        # Statistical summary
        summary = {
            'matched_dates': candidates.index[:10].tolist(),
            'outcomes': outcomes_df,
            'statistics': outcomes_df.describe(),
            'current_state': current_state
        }
        
        return summary
    
    def _generate_outputs(self, var_res: Dict, irf_results: Dict, 
                         regime_results: Dict, scenarios: Dict, var_df: pd.DataFrame):
        """Generate comprehensive outputs and visualizations"""
        
        # Save numerical results
        dump(var_res, os.path.join(self.config.OUTDIR, "var_results.joblib"))
        dump(regime_results, os.path.join(self.config.OUTDIR, "regime_results.joblib"))
        np.save(os.path.join(self.config.OUTDIR, "irf_full.npy"), irf_results['irf'])
        
        # Save data
        var_df.to_csv(os.path.join(self.config.OUTDIR, "var_dataset.csv"))
        scenarios['outcomes'].to_csv(os.path.join(self.config.OUTDIR, "scenario_outcomes.csv"))
        
        # Generate visualizations
        self._plot_irfs(irf_results, var_df.columns)
        self._plot_regimes(regime_results, var_df)
        self._plot_scenario_distribution(scenarios)
        
        # Generate summary report
        self._generate_summary_report(var_res, irf_results, regime_results, scenarios)
    
    def _plot_irfs(self, irf_results: Dict, var_names: List[str]):
        """Plot IRFs with confidence bands"""
        
        irfs = irf_results['irf']
        lower = irf_results['lower']
        upper = irf_results['upper']
        
        # Select key IRFs to plot
        n_vars = len(var_names)
        fig, axes = plt.subplots(3, 3, figsize=(15, 12))
        axes = axes.flatten()
        
        plot_idx = 0
        for i in range(min(3, n_vars)):  # Response variables
            for j in range(min(3, n_vars)):  # Shock variables
                ax = axes[plot_idx]
                
                # Plot IRF with confidence band
                horizon = np.arange(irfs.shape[0])
                ax.plot(horizon, irfs[:, i, j], 'b-', label='IRF')
                ax.fill_between(horizon, lower[:, i, j], upper[:, i, j], 
                               alpha=0.3, color='blue', label='95% CI')
                ax.axhline(0, color='black', linestyle='--', alpha=0.5)
                
                ax.set_title(f'{var_names[i]} ← {var_names[j]}')
                ax.set_xlabel('Horizon (days)')
                ax.grid(True, alpha=0.3)
                
                if plot_idx == 0:
                    ax.legend()
                
                plot_idx += 1
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.OUTDIR, "irf_matrix.png"), dpi=150)
        plt.close()
    
    def _plot_regimes(self, regime_results: Dict, var_df: pd.DataFrame):
        """Plot regime identification"""
        
        regimes = regime_results['regimes']
        
        fig, axes = plt.subplots(2, 1, figsize=(15, 10))
        
        # Plot regimes over time
        axes[0].plot(var_df.index[:len(regimes)], regimes)
        axes[0].set_title('Identified Market Regimes')
        axes[0].set_ylabel('Regime')
        axes[0].grid(True, alpha=0.3)
        
        # Plot transition matrix as heatmap
        trans_mat = regime_results['transition_matrix']
        sns.heatmap(trans_mat, annot=True, fmt='.3f', cmap='YlOrRd', 
                   ax=axes[1], cbar_kws={'label': 'Probability'})
        axes[1].set_title('Regime Transition Probabilities')
        axes[1].set_xlabel('To Regime')
        axes[1].set_ylabel('From Regime')
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.OUTDIR, "regimes.png"), dpi=150)
        plt.close()
    
    def _plot_scenario_distribution(self, scenarios: Dict):
        """Plot scenario analysis results"""
        
        outcomes = scenarios['outcomes']
        
        fig, axes = plt.subplots(2, 2, figsize=(15, 10))
        axes = axes.flatten()
        
        # Plot return distributions at different horizons
        horizons = ['SPY_21d', 'SPY_63d', 'SPY_126d', 'SPY_252d']
        
        for i, col in enumerate(horizons):
            if col in outcomes.columns:
                ax = axes[i]
                
                # Histogram with KDE
                outcomes[col].dropna().hist(bins=20, ax=ax, alpha=0.6, color='blue')
                outcomes[col].dropna().plot.kde(ax=ax, secondary_y=True, color='red')
                
                # Add statistics
                mean = outcomes[col].mean()
                median = outcomes[col].median()
                ax.axvline(mean, color='green', linestyle='--', label=f'Mean: {mean:.2%}')
                ax.axvline(median, color='orange', linestyle='--', label=f'Median: {median:.2%}')
                
                ax.set_title(f'Forward Returns: {col}')
                ax.set_xlabel('Return')
                ax.legend()
                ax.grid(True, alpha=0.3)
        
        plt.tight_layout()
        plt.savefig(os.path.join(self.config.OUTDIR, "scenario_distributions.png"), dpi=150)
        plt.close()
    
    def _generate_summary_report(self, var_res: Dict, irf_results: Dict, 
                                regime_results: Dict, scenarios: Dict):
        """Generate comprehensive text summary"""
        
        report = []
        report.append("=" * 60)
        report.append("MACRO-FINANCIAL VAR ANALYSIS SUMMARY")
        report.append("=" * 60)
        report.append(f"Report Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
        
        # Model diagnostics
        report.append("MODEL DIAGNOSTICS")
        report.append("-" * 30)
        report.append(f"VAR Lags: {self.config.VAR_LAGS}")
        report.append(f"Number of Factors: {self.config.N_FACTORS}")
        
        if self.config.USE_BAYESIAN_VAR:
            report.append(f"Method: Bayesian VAR (Minnesota Prior)")
            report.append(f"Prior Tightness: {self.config.MINNESOTA_PRIOR_TIGHTNESS}")
        else:
            report.append(f"Method: Penalized VAR (Ridge/ElasticNet)")
        
        # Residual diagnostics
        residuals = var_res['residuals']
        report.append(f"\nResidual Standard Deviations:")
        for i in range(min(5, residuals.shape[1])):
            report.append(f"  Variable {i+1}: {np.std(residuals[:, i]):.4f}")
        
        # Regime analysis
        report.append("\nREGIME ANALYSIS")
        report.append("-" * 30)
        regimes = regime_results['regimes']
        unique, counts = np.unique(regimes, return_counts=True)
        for r, c in zip(unique, counts):
            report.append(f"Regime {r}: {c} periods ({100*c/len(regimes):.1f}%)")
        
        # Scenario analysis
        report.append("\nSCENARIO ANALYSIS")
        report.append("-" * 30)
        report.append("Similar Historical Episodes:")
        for date in scenarios['matched_dates'][:5]:
            report.append(f"  {date.strftime('%Y-%m-%d')}")
        
        report.append("\nExpected Forward Returns (based on historical matches):")
        stats = scenarios['statistics']
        for col in ['SPY_21d', 'SPY_63d', 'SPY_252d']:
            if col in stats.columns:
                mean = stats.loc['mean', col]
                std = stats.loc['std', col]
                report.append(f"  {col}: {mean:.2%} (±{std:.2%})")
        
        # Risk metrics
        report.append("\nRISK METRICS")
        report.append("-" * 30)
        
        # Calculate VaR from scenario distribution
        if 'SPY_252d' in scenarios['outcomes'].columns:
            returns = scenarios['outcomes']['SPY_252d'].dropna()
            var_95 = np.percentile(returns, 5)
            var_99 = np.percentile(returns, 1)
            report.append(f"1-Year VaR (95%): {var_95:.2%}")
            report.append(f"1-Year VaR (99%): {var_99:.2%}")
        
        # Key economic relationships (from IRF)
        report.append("\nKEY ECONOMIC RELATIONSHIPS")
        report.append("-" * 30)
        report.append("Peak IRF responses (standardized shocks):")
        
        irf = irf_results['irf']
        # Find peak responses for key relationships
        if irf.shape[1] >= 3 and irf.shape[2] >= 3:
            cpi_to_rates = np.max(np.abs(irf[:, 3, 0])) if irf.shape[1] > 3 else 0
            report.append(f"  CPI → 10Y Yield: {cpi_to_rates:.3f}")
        
        # Save report
        report_text = "\n".join(report)
        with open(os.path.join(self.config.OUTDIR, "summary_report.txt"), 'w') as f:
            f.write(report_text)
        
        logger.info("Summary report generated")
        print("\n" + report_text)


In [ ]:

# ========== EXECUTION ==========
def main():
    """Main execution function"""
    
    try:
        # Initialize configuration
        config = Config()
        
        # Run pipeline
        pipeline = MacroVARPipeline(config)
        results = pipeline.run()
        
        logger.info("Pipeline completed successfully!")
        logger.info(f"Results saved to {config.OUTDIR}/")
        
        return results
        
    except Exception as e:
        logger.error(f"Pipeline failed: {str(e)}", exc_info=True)
        raise

if __name__ == "__main__":
    results = main()